In [ ]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# -----------------------------
# 1. Load dataset
# -----------------------------
df = pd.read_csv("emails.csv")
df = df.dropna()

X = df["text"]
y = df["label"]

print("Label counts:")
print(df["label"].value_counts())
print()

# -----------------------------
# 2. Split data
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# -----------------------------
# 3. Vectorize text
# -----------------------------
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# -----------------------------
# 4. Train model
# -----------------------------
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# -----------------------------
# 5. Evaluate model
# -----------------------------
y_pred = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.2f}")
print()

# -----------------------------
# 6. Rule-based analysis
# -----------------------------
def analyze_rules(text):
    text_lower = text.lower()
    score = 0
    reasons = []

    suspicious_keywords = [
        "urgent", "verify", "password", "account", "bank",
        "click", "login", "confirm", "suspend", "locked",
        "security", "alert", "free", "winner", "claim",
        "immediately", "address", "invoice", "package"
    ]

    for word in suspicious_keywords:
        if word in text_lower:
            score += 1
            reasons.append(f"Suspicious keyword: '{word}'")

    # Link detection
    if "http://" in text_lower or "https://" in text_lower or "www." in text_lower:
        score += 2
        reasons.append("Contains a link")

    # Email detection
    email_pattern = r'[\w\.-]+@[\w\.-]+\.\w+'
    if re.search(email_pattern, text):
        score += 1
        reasons.append("Contains email address")

    # Relay detection
    if "privaterelay.appleid.com" in text_lower:
        score += 1
        reasons.append("Uses Apple private relay address")

    # Sender style format
    if "<" in text and ">" in text and "@" in text:
        score += 1
        reasons.append("Formatted like sender display email")

    return score, reasons

# -----------------------------
# 7. Risk score calculation
# -----------------------------
def calculate_risk_percentage(ml_prediction, rule_score):
    risk = 10

    if ml_prediction == "phishing":
        risk += 45
    else:
        risk += 10

    risk += rule_score * 10

    if risk > 100:
        risk = 100

    return risk

# -----------------------------
# 8. Final verdict
# -----------------------------
def final_verdict(risk):
    if risk >= 75:
        return "🚨 High Risk"
    elif risk >= 45:
        return "⚠️ Suspicious"
    else:
        return "✅ Likely Safe"

# -----------------------------
# 9. Analyze one message
# -----------------------------
def analyze_message(user_text):
    user_vec = vectorizer.transform([user_text])
    ml_prediction = model.predict(user_vec)[0]

    rule_score, reasons = analyze_rules(user_text)
    risk = calculate_risk_percentage(ml_prediction, rule_score)
    verdict = final_verdict(risk)

    print("\n==============================")
    print("Message:")
    print(user_text)
    print("------------------------------")
    print("ML Prediction :", ml_prediction)
    print("Rule Score    :", rule_score)
    print("Risk Score    :", f"{risk}%")
    print("Final Verdict :", verdict)

    if reasons:
        print("\nReasons:")
        for reason in reasons:
            print("-", reason)
    else:
        print("\nNo suspicious indicators found.")

    print("==============================\n")

# -----------------------------
# 10. Main loop
# -----------------------------
print("Phishing Detection Tool")
print("Type a message to analyze.")
print("Type 'exit' to quit.\n")

while True:
    user_text = input("Enter email text or sender:\n> ")

    if user_text.lower() == "exit":
        print("Goodbye.")
        break

    if not user_text.strip():
        print("Please enter some text.\n")
        continue

    analyze_message(user_text)

Label counts:
label
phishing    8
safe        8
Name: count, dtype: int64

Model accuracy: 0.80

Phishing Detection Tool
Type a message to analyze.
Type 'exit' to quit.




Message:
Koji <notifications_at_account_brilliant_org_th279fsyn9_787a9855@privaterelay.appleid.com‏>
------------------------------
ML Prediction : safe
Rule Score    : 4
Risk Score    : 60%
Final Verdict : ⚠️ Suspicious

Reasons:
- Suspicious keyword: 'account'
- Contains email address
- Uses Apple private relay address
- Formatted like sender display email

Please enter some text.

Please enter some text.

Please enter some text.

Please enter some text.

